# Práctica: Regresión Lineal — Predicción de Precios de Vivienda
### Dataset: California Housing (Scikit-Learn)

En esta práctica se cubre: obtención y preprocesamiento de datos, análisis de variables, separación entrenamiento/prueba, implementación del modelo, evaluación con métricas de regresión, y predicción con datos nuevos.

In [1]:
## Importamos las librerías necesarias para el análisis y el modelo de ML
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing # descarga y carga el dataset 
from sklearn.model_selection import train_test_split #divicion 
from sklearn.preprocessing import StandardScaler  # escaldo
from sklearn.linear_model import LinearRegression # importa el algoritmo 
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score   # importa las metricas 

In [ ]:
## Descargamos el dataset California Housing directamente desde Scikit-Learn
## Fuente: repositorio interno de sklearn.datasets
datos = fetch_california_housing(as_frame=True)
df = datos.frame

## Visualizamos las primeras filas para confirmar que se cargó correctamente
df.head()

In [ ]:
## Revisamos información general: tipos de datos y cantidad de registros
df.info()

In [ ]:
## Verificamos si existen valores nulos (parte del preprocesamiento)
## Este dataset viene limpio de fábrica, pero siempre se debe comprobar
df.isnull().sum()

In [ ]:
## Revisamos estadísticas descriptivas para detectar posibles valores atípicos (outliers)
df.describe()

## 2. Importancia de las variables

In [ ]:
## Separamos las variables predictoras (X) de la variable objetivo (y)
## Variable objetivo: MedHouseVal (precio medio de la vivienda, en cientos de miles de USD)
## Variables predictoras: el resto de columnas (ingreso medio, antigüedad, habitaciones, etc.)
X = df.drop(columns=['MedHouseVal'])
y = df['MedHouseVal']

print("Variables predictoras:", list(X.columns))
print("Variable objetivo: MedHouseVal")

In [ ]:
## Calculamos la correlación de cada variable predictora con la variable objetivo
## Esto nos da una primera idea de qué variables son más relevantes
correlaciones = df.corr()['MedHouseVal'].drop('MedHouseVal').sort_values(ascending=False)
correlaciones

In [ ]:
## Graficamos la correlación de cada variable con el precio de vivienda
## (una forma simple y válida de mostrar "importancia de variables" antes de entrenar)
plt.figure(figsize=(8,5))
correlaciones.plot(kind='barh', color='steelblue')
plt.title('Correlación de cada variable con el precio de vivienda (MedHouseVal)')
plt.xlabel('Coeficiente de correlación')
plt.tight_layout()
plt.show()

## 3. Separación de datos: entrenamiento y prueba

In [ ]:
## Separamos los datos en conjunto de entrenamiento (80%) y prueba (20%)
## random_state=42 fija la semilla para que la división sea reproducible
## No se usa estratificación porque el target es continuo (la estratificación aplica a clases, no a valores numéricos)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Tamaño de entrenamiento:", X_train.shape)
print("Tamaño de prueba:", X_test.shape)

In [ ]:
## Escalado de variables: ajustamos el scaler SOLO con datos de entrenamiento
## y lo aplicamos (transform) tanto a entrenamiento como a prueba
## Esto evita fuga de información del conjunto de prueba hacia el entrenamiento
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit + transform en train
X_test_scaled = scaler.transform(X_test)          # solo transform en test

## 4. Implementación del modelo

In [ ]:
## Creamos y entrenamos el modelo de Regresión Lineal
## Librería: sklearn.linear_model.LinearRegression
## Parámetro relevante: fit_intercept=True (por defecto), permite que el modelo calcule el bias/intercepto
modelo = LinearRegression(fit_intercept=True)
modelo.fit(X_train_scaled, y_train)

print("Modelo entrenado correctamente")

In [ ]:
## Revisamos los coeficientes aprendidos (uno por cada variable predictora) y el intercepto
coeficientes = pd.DataFrame({
    'variable': X.columns,
    'coeficiente': modelo.coef_
}).sort_values('coeficiente', key=abs, ascending=False)

print("Intercepto (beta_0):", modelo.intercept_)
coeficientes

In [ ]:
## Generamos las predicciones sobre el conjunto de prueba
y_pred = modelo.predict(X_test_scaled)

## Mostramos las primeras predicciones comparadas contra el valor real
comparacion = pd.DataFrame({'Real': y_test.values[:10], 'Predicho': y_pred[:10]})
comparacion

## 5. Evaluación del modelo

In [ ]:
## Calculamos las métricas de evaluación adecuadas para un problema de REGRESIÓN
##  aquí el target es un valor numérico continuo, no una categoría/clase)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"MAE  (Error Absoluto Medio):     {mae:.3f}")
print(f"MSE  (Error Cuadrático Medio):   {mse:.3f}")
print(f"RMSE (Raíz del Error Cuadrático): {rmse:.3f}")
print(f"R²   (Coeficiente de determinación): {r2:.3f}")

In [ ]:
## Graficamos valores reales vs. predichos para visualizar el desempeño del modelo
plt.figure(figsize=(6,6))
plt.scatter(y_test, y_pred, alpha=0.3, color='teal')
plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--')  # línea de referencia "predicción perfecta"
plt.xlabel('Valor real (MedHouseVal)')
plt.ylabel('Valor predicho')
plt.title('Predicción vs. Valor Real')
plt.tight_layout()
plt.show()

In [ ]:
## Graficamos los residuos (errores) para verificar si el modelo lineal es adecuado
residuos = y_test - y_pred
plt.figure(figsize=(6,4))
plt.scatter(y_pred, residuos, alpha=0.3, color='indianred')
plt.axhline(y=0, color='black', linestyle='--')
plt.xlabel('Valor predicho')
plt.ylabel('Residuo (Real - Predicho)')
plt.title('Gráfico de residuos')
plt.tight_layout()
plt.show()

In [ ]:
## Probamos el modelo con un dato COMPLETAMENTE nuevo (inventado), simulando un caso real
## El formato de entrada debe tener EXACTAMENTE las mismas columnas y orden que X_train

nuevo_dato = pd.DataFrame({
    'MedInc': [5.5],       # ingreso medio de la zona (en decenas de miles de USD)
    'HouseAge': [15],      # antigüedad de la vivienda en años
    'AveRooms': [6.0],     # promedio de habitaciones por casa
    'AveBedrms': [1.1],    # promedio de dormitorios por casa
    'Population': [1200],  # población del bloque censal
    'AveOccup': [3.0],     # ocupación promedio por vivienda
    'Latitude': [34.05],   # ubicación (ej. cerca de Los Ángeles)
    'Longitude': [-118.25]
})

nuevo_dato

In [ ]:
## IMPORTANTE: aplicamos el MISMO preprocesamiento usado en entrenamiento (mismo scaler, ya ajustado)
## Nunca se debe volver a hacer fit() aquí, solo transform()
nuevo_dato_scaled = scaler.transform(nuevo_dato)

## Generamos la predicción
prediccion_nueva = modelo.predict(nuevo_dato_scaled)

print(f"Precio medio de vivienda predicho: {prediccion_nueva[0]:.3f} (en cientos de miles de USD)")
print(f"Equivalente aproximado: ${prediccion_nueva[0]*100000:,.2f} USD")

## CONCLUCIONES 

La regresión lineal es un algoritmo fundamental del Machine Learning porque, con un modelo matemático simple, resuelve un problema clave: predecir un valor numérico a partir de datos históricos. Es útil porque es interpretable (cada coeficiente explica claramente el efecto de cada variable), rápido de entrenar, y sirve como punto de referencia antes de usar modelos más complejos